In [31]:
import pandas as pd

In [32]:
train = pd.read_csv("C:/Users/gadha/Downloads/archive (51)/SMS_train.csv", encoding='latin-1')
test = pd.read_csv("C:/Users/gadha/Downloads/archive (51)/SMS_test.csv", encoding='latin-1')

In [33]:
train.head()

,S. No.,Message_body,Label
0,1,Rofl. Its true to its name,Non-Spam
1,2,The guy did some bitching but I acted like i'd...,Non-Spam
2,3,"Pity, * was in mood for that. So...any other s...",Non-Spam
3,4,Will ü b going to esplanade fr home?,Non-Spam
4,5,This is the 2nd time we have tried 2 contact u...,Spam


In [34]:
import re

In [35]:
def clean(text):
    text = text.lower()
    text = re.sub(r"[^\w\s]", " ", text)   
    text = re.sub(r"\s+", " ", text).strip()
    return text
train["Message_body"] = train["Message_body"].apply(clean)
test["Message_body"] = test["Message_body"].apply(clean)

In [36]:
train

,S. No.,Message_body,Label
0,1,rofl its true to its name,Non-Spam
1,2,the guy did some bitching but i acted like i d...,Non-Spam
2,3,pity was in mood for that so any other suggest...,Non-Spam
3,4,will ü b going to esplanade fr home,Non-Spam
4,5,this is the 2nd time we have tried 2 contact u...,Spam
...,...,...,...
952,953,hows my favourite person today r u workin hard...,Non-Spam
953,954,how much you got for cleaning,Non-Spam
954,955,sorry da i gone mad so many pending works what...,Non-Spam
955,956,wat time ü finish,Non-Spam


In [37]:
test

,S. No.,Message_body,Label
0,1,upgrdcentre orange customer you may now claim ...,Spam
1,2,loan for any purpose 500 75 000 homeowners ten...,Spam
2,3,congrats nokia 3650 video camera phone is your...,Spam
3,4,urgent your mobile number has been awarded wit...,Spam
4,5,someone has contacted our dating service and e...,Spam
...,...,...,...
120,121,7 wonders in my world 7th you 6th ur style 5th...,Non-Spam
121,122,try to do something dear you read something fo...,Non-Spam
122,123,sun ah thk mayb can if dun have anythin on thk...,Non-Spam
123,124,symptoms when u are in love 1 u like listening...,Non-Spam


In [38]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()

train["Label"] = le.fit_transform(train["Label"])
test["Label"] = le.transform(test["Label"])

In [39]:
from tensorflow.keras.preprocessing.text import Tokenizer
max_words = 5000
tokenizer = Tokenizer(num_words=max_words)
tokenizer.fit_on_texts(train["Message_body"])

x_train = tokenizer.texts_to_sequences(train["Message_body"])
x_test = tokenizer.texts_to_sequences(test["Message_body"])

In [40]:
from nltk.corpus import stopwords
stop_words = set(stopwords.words('english'))

def clean(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

In [42]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
max_len = 100

x_train = pad_sequences(x_train, maxlen=max_len, padding='post')
x_test = pad_sequences(x_test, maxlen=max_len, padding='post')

y_train = train["Label"]
y_test = test["Label"]

In [57]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Bidirectional

In [58]:
model = Sequential()
model.add(Embedding(input_dim=5000, output_dim=64, input_length=max_len))
model.add(Bidirectional(LSTM(64, return_sequences=True)))
model.add(Dropout(0.5))
model.add(Bidirectional(LSTM(32)))
model.add(Dropout(0.5))
model.add(Dense(1, activation='sigmoid'))
model.compile(loss='binary_crossentropy',optimizer='adam',metrics=['accuracy'])

In [60]:
history = model.fit(x_train, y_train,epochs=20,batch_size=32,validation_data=(x_test, y_test))

Epoch 1/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 1.0000 - loss: 5.8271e-04 - val_accuracy: 0.9200 - val_loss: 0.5168
Epoch 2/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 94ms/step - accuracy: 1.0000 - loss: 5.6869e-04 - val_accuracy: 0.9280 - val_loss: 0.5230
Epoch 3/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 101ms/step - accuracy: 1.0000 - loss: 5.0188e-04 - val_accuracy: 0.9280 - val_loss: 0.5289
Epoch 4/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 99ms/step - accuracy: 1.0000 - loss: 4.1508e-04 - val_accuracy: 0.9280 - val_loss: 0.5304
Epoch 5/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 5s 96ms/step - accuracy: 1.0000 - loss: 5.0190e-04 - val_accuracy: 0.9280 - val_loss: 0.5320
Epoch 6/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 104ms/step - accuracy: 1.0000 - loss: 6.1514e-04 - val_accuracy: 0.9360 - val_loss: 0.5246
Epoch 7/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 102ms/step - accuracy: 1.0000 - loss: 4.5687e-04 - val_accuracy: 0.9360 - val_loss: 0.5268
Epoch 8/20
30/30 ━━━━━━━━━━━━━━━━━━━━ 3s 112ms/step - accuracy: 1.0000 - loss: 

In [61]:
loss, acc = model.evaluate(x_test, y_test)
print("Accuracy:", acc)

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.9120 - loss: 0.7330
Accuracy: 0.9120000004768372


In [62]:
def predict_spam(text):
    text = clean(text)
    seq = tokenizer.texts_to_sequences([text])
    padded = pad_sequences(seq, maxlen=max_len)  
    pred = model.predict(padded)[0][0]   
    return "Spam" if pred > 0.5 else "Not Spam"
print(predict_spam("Congratulations! You won a free prize"))

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 931ms/step
Not Spam


In [63]:
import pickle
model.save("model.h5")
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)